In [74]:
import pandas as pd
import numpy as np

In [160]:
df = pd.read_csv('data/B208_2022-10-09_03-26-03_2022-10-09_22-56-13.csv')

In [161]:
# convert radians to degrees
df['latitude'] = df['gnss_latitude'] * 180.0 / np.pi
df['longitude'] = df['gnss_longitude'] * 180.0 / np.pi
# convert temperature to celsius
df['temp_celsius'] = df['temperature_ambient'] - 273.15
# Convert to datetime and strip the +00:00 timezone
df['timestamp'] = pd.to_datetime(df['time_iso']).dt.tz_localize(None)
# add an extra column that shows the speed in km
df['speed_kmh'] = df['odometry_vehicleSpeed'] * 3.6

# convert compass course to degrees and If speed is near zero on Bus 208, keep the last known heading instead of snapping to 0 (useful for UI later)
df['heading_deg'] = df['gnss_course'] * 180.0 / np.pi
df.loc[df['odometry_vehicleSpeed'] < 0.5, 'heading_deg'] = np.nan
df['heading_deg'] = df['heading_deg'].ffill().fillna(0).round(1)

# create a column with power in Kilowatts
df['power_kw'] = (df['electric_powerDemand'] / 1000.0).round(2)

# brake pressure in bars
df['brake_bar'] = (df['traction_brakePressure'] / 100000.0).round(2)

# ffill passenger count so they persist while driving
df['itcs_numberOfPassengers'] = df['itcs_numberOfPassengers'].ffill().fillna(0).astype(int)


df = df.drop(columns=['gnss_latitude', 'gnss_longitude', 'temperature_ambient', 'time_iso', 'gnss_course'])
df.sample(100)

,time_unix,electric_powerDemand,gnss_altitude,itcs_busRoute,itcs_numberOfPassengers,itcs_stopName,odometry_articulationAngle,odometry_steeringAngle,odometry_vehicleSpeed,odometry_wheelSpeed_fl,...,traction_brakePressure,traction_tractionForce,latitude,longitude,temp_celsius,timestamp,speed_kmh,heading_deg,power_kw,brake_bar
23546,1665309509,22769.61,420.9866,-,28,-,0.043633,0.008727,13.039280,12.847570,...,0.0,1894.1220,47.392233,8.524059,15.0,2022-10-09 09:58:29,46.941408,30.1,22.77,0.00
70015,1665355978,71406.22,415.8216,-,2,-,0.052360,0.000000,11.322380,11.146140,...,0.0,5274.9490,47.381341,8.510715,14.0,2022-10-09 22:52:58,40.760568,302.1,71.41,0.00
6746,1665292709,2223.23,434.7346,-,3,-,0.061087,0.000000,10.554920,10.452300,...,0.0,-109.3768,47.371159,8.493108,12.0,2022-10-09 05:18:29,37.997712,36.9,2.22,0.00
24273,1665310236,26032.24,538.0000,-,16,-,0.052360,0.000000,0.301060,0.000000,...,0.0,27679.7200,47.379272,8.559136,14.0,2022-10-09 10:10:36,1.083816,129.6,26.03,0.00
4653,1665290616,-30089.11,520.0000,-,4,-,0.026180,0.017453,3.761497,3.741729,...,0.0,-11819.4000,47.377238,8.559216,13.0,2022-10-09 04:43:36,13.541389,314.6,-30.09,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57085,1665343048,-69919.93,445.0346,-,7,-,0.061087,0.000000,7.141319,7.114232,...,0.0,-17067.8300,47.362364,8.561256,15.0,2022-10-09 19:17:28,25.708748,244.5,-69.92,0.00
55750,1665341713,41996.31,421.0844,-,16,-,0.052360,0.052360,1.664449,1.691329,...,0.0,16053.7700,47.385271,8.517338,16.0,2022-10-09 18:55:13,5.992016,35.6,42.00,0.00
6169,1665292132,21588.61,453.9000,-,6,-,0.087267,0.000000,0.000000,0.000000,...,240833.3,0.0000,47.368105,8.495686,12.0,2022-10-09 05:08:52,0.000000,79.6,21.59,2.41
37438,1665323401,54892.73,436.7091,-,54,-,0.104720,-0.061087,7.909411,7.862249,...,0.0,6560.7730,47.394238,8.525618,16.0,2022-10-09 13:50:01,28.473880,287.6,54.89,0.00


In [169]:
# Slice out just the 7:00 AM to 9:00 AM window for tests
df_demo = df[(df['timestamp'].dt.hour >= 7) & (df['timestamp'].dt.hour <= 8)]

In [171]:
df_demo.to_csv('data/clean_replay_mission.csv', index=False)
df_demo.head(100).to_json('data/mock_state.json', orient='records', indent=2)
len(df_demo)

7200

,time_unix,electric_powerDemand,gnss_altitude,itcs_busRoute,itcs_numberOfPassengers,itcs_stopName,odometry_articulationAngle,odometry_steeringAngle,odometry_vehicleSpeed,odometry_wheelSpeed_fl,...,traction_brakePressure,traction_tractionForce,latitude,longitude,temp_celsius,timestamp,speed_kmh,heading_deg,power_kw,brake_bar
12837,1665298800,3500.530,428.4076,-,18,-,0.008727,0.017453,0.000000,0.000000,...,245833.3,0.00000,47.394525,8.525618,13.0,2022-10-09 07:00:00,0.000000,26.2,3.50,2.46
12838,1665298801,3642.494,428.3077,-,18,-,0.008727,0.017453,0.000000,0.000000,...,246666.7,0.00000,47.394525,8.525618,13.0,2022-10-09 07:00:01,0.000000,26.2,3.64,2.47
12839,1665298802,3615.297,428.2075,-,18,-,0.008727,0.017453,0.000000,0.000000,...,248333.3,0.00000,47.394525,8.525623,13.0,2022-10-09 07:00:02,0.000000,26.2,3.62,2.48
12840,1665298803,3519.582,428.1073,-,18,-,0.008727,0.017453,0.000000,0.000000,...,248333.3,0.00000,47.394525,8.525623,13.0,2022-10-09 07:00:03,0.000000,26.2,3.52,2.48
12841,1665298804,3429.395,428.0076,-,18,-,0.008727,0.017453,0.000000,0.000000,...,248333.3,0.00000,47.394525,8.525623,13.0,2022-10-09 07:00:04,0.000000,26.2,3.43,2.48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20032,1665305995,15677.810,412.8130,-,3,-,-0.008727,0.069813,8.484355,8.297482,...,0.0,1265.96400,47.352378,8.559663,14.0,2022-10-09 08:59:55,30.543678,331.8,15.68,0.00
20033,1665305996,4886.899,412.8000,-,3,-,-0.008727,0.043633,8.375529,8.223783,...,0.0,-21.96729,47.352446,8.559606,14.0,2022-10-09 08:59:56,30.151904,328.0,4.89,0.00
20034,1665305997,5224.254,412.8000,-,3,-,0.014897,0.011283,8.249250,8.133681,...,0.0,-12.80313,47.352504,8.559554,14.0,2022-10-09 08:59:57,29.697300,327.4,5.22,0.00
20035,1665305998,20703.640,412.6134,-,3,-,0.026180,0.008727,8.154318,8.051158,...,0.0,1962.10300,47.352567,8.559491,14.0,2022-10-09 08:59:58,29.355545,326.5,20.70,0.00
